# Attention and Transformers: in-class activities
**MATH60629A — Machine Learning I — Week 10**

This notebook goes with the week's slides. It contains three activities:

1. **The bottleneck game** (≈ 10 min): why an RNN encoder-decoder struggles with long sequences.
2. **Positional embeddings** (≈ 10 min): what self-attention "sees" when you change the word order.
3. **Multi-head attention** (≈ 15 min): several heads, several notions of similarity; exploring a real model (BERT).

Run the cells in order. Cells marked **Solution** are collapsed: double-click to see the code, or run them to see the result.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

def softmax(S, axis=-1):
    S = S - S.max(axis=axis, keepdims=True)  # numerical stability
    E = np.exp(S)
    return E / E.sum(axis=axis, keepdims=True)

---
## 1. The bottleneck game

In an RNN encoder-decoder, the **entire** source sentence has to pass through a single fixed-size vector (the final hidden state $h_T$) before the decoder can produce the translation. We are going to play the role of that vector.

**Rules**

1. Form teams of two: the **encoder** (A) and the **decoder** (B).
2. The decoder looks away. The encoder reads the projected paragraph (next cell) for one minute.
3. The encoder summarizes the paragraph in **exactly 5 words** and passes them to the decoder (nothing else: no gestures, no creative punctuation!).
4. The decoder reconstructs the paragraph as faithfully as possible, in two minutes.
5. Compare. Which information survived? What disappeared first?

*Variant:* repeat with 15 words. Then with 5 words, but let the decoder ask the encoder one targeted question **for each sentence it writes**. That is exactly what attention does.

In [ ]:
#@title Paragraph to project (encoders only)
paragraph = (
    "William Shakespeare was born in Stratford-upon-Avon in 1564, the son of a glove maker. "
    "At eighteen he married Anne Hathaway, and by the early 1590s he was working in London as "
    "an actor and playwright. He became a part-owner of the Lord Chamberlain's Men, the company "
    "that built the Globe Theatre on the south bank of the Thames in 1599. Over about twenty-five "
    "years he wrote close to forty plays, including Hamlet, Macbeth and A Midsummer Night's Dream, "
    "as well as more than 150 sonnets. He retired to Stratford and died there in 1616."
)
print(paragraph)

In [ ]:
# Encoder (A): exactly 5 words
summary = ["", "", "", "", ""]

# Decoder (B): your reconstruction
reconstruction = '''
'''

In [ ]:
#@title Compare the reconstruction with the original
import re

def content_words(text):
    # Words longer than 3 letters, lowercased: a rough measure of content
    return {w for w in re.findall(r"[\w'-]+", text.lower()) if len(w) > 3}

assert len([w for w in summary if w.strip()]) <= 5, "The summary must have at most 5 words!"
original = content_words(paragraph)
recovered = original & content_words(reconstruction)
print(f"Content words recovered: {len(recovered)} / {len(original)}")
print("Recovered:", sorted(recovered))
print("Lost     :", sorted(original - recovered))

### The same phenomenon, in numbers

A simple experiment: each word in a 5,000-word vocabulary gets a random vector of dimension $d$. We "encode" a sequence of $n$ words by **summing** their vectors (a fixed-size vector, like $h_T$). Then we try to recover which words were in the sequence by finding the $n$ vectors most similar to the summary.

This encoder is very simplified (it even ignores word order, which makes the task *easier*!), but it illustrates the capacity constraint.

**Before running:** how will the recall change as $n$ grows? And as $d$ grows?

In [ ]:
def mean_recall(d, n, V=5000, trials=200, seed=0):
    rng = np.random.default_rng(seed)
    E = rng.normal(size=(V, d)) / np.sqrt(d)   # one vector per vocabulary word
    total = 0.0
    for _ in range(trials):
        words = rng.choice(V, size=n, replace=False)
        h = E[words].sum(axis=0)                # the "bottleneck": a single vector of size d
        scores = E @ h
        guesses = np.argpartition(-scores, n)[:n]
        total += len(set(guesses) & set(words)) / n
    return total / trials

lengths = [1, 2, 5, 10, 20, 50, 100]
plt.figure(figsize=(7, 4))
for d in [16, 64, 256]:
    plt.plot(lengths, [mean_recall(d, n) for n in lengths], marker="o", label=f"d = {d}")
plt.xscale("log")
plt.xlabel("Sequence length n")
plt.ylabel("Fraction of words recovered")
plt.title("A fixed-size vector cannot remember everything")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Discussion**

* For a given size $d$, at what length does reconstruction start to degrade?
* Increasing $d$ helps, but at what cost? (Think of the number of parameters in an RNN, which grows as $d^2$.)
* With attention, the decoder has access to **all** the $h_j$: the representation grows with the sequence. What would this curve look like then?

---
## 2. Positional embeddings

Take two sentences that contain **exactly the same words**, in a different order:

* "the cat ate the plant"
* "the plant ate the cat"

We pass each one through a self-attention layer (random weights, $D = 8$), **without** positional embeddings.

**Before running:** will the output representation of "cat" be the same in both sentences? What about the mean representation of the sentence?

In [ ]:
rng = np.random.default_rng(42)
D = 8
vocabulary = ["the", "cat", "ate", "plant"]
embedding = {w: rng.normal(size=D) for w in vocabulary}
Wq, Wk, Wv = (rng.normal(size=(D, D)) / np.sqrt(D) for _ in range(3))

def self_attention(X):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    A = softmax(Q @ K.T / np.sqrt(X.shape[1]), axis=1)
    return A @ V

sentence1 = "the cat ate the plant".split()
sentence2 = "the plant ate the cat".split()

def compare(Z1, Z2, title):
    print(title)
    for w in ["cat", "ate", "plant"]:
        z1 = Z1[sentence1.index(w)]
        z2 = Z2[sentence2.index(w)]
        print(f"  {w:>6}: difference between the two sentences = {np.abs(z1 - z2).max():.2e}")
    print(f"  Mean sentence representation: difference = {np.abs(Z1.mean(0) - Z2.mean(0)).max():.2e}\n")

X1 = np.stack([embedding[w] for w in sentence1])
X2 = np.stack([embedding[w] for w in sentence2])
compare(self_attention(X1), self_attention(X2), "Without positional embeddings:")

The differences are zero (up to rounding error): without position information, self-attention treats a sentence as a **set** of words. Reordering the input words only reorders the outputs; we say self-attention is *permutation equivariant*. To it, "the cat ate the plant" and "the plant ate the cat" are the same thing.

**The fix:** add to each word a vector that depends on its position. Here we use the sinusoidal embeddings from Vaswani et al. (many models learn them instead).

In [ ]:
def positional_embeddings(n, D):
    pos = np.arange(n)[:, None]
    i = np.arange(0, D, 2)[None, :]
    angles = pos / (10000 ** (i / D))
    P = np.zeros((n, D))
    P[:, 0::2] = np.sin(angles)
    P[:, 1::2] = np.cos(angles)
    return P

P = positional_embeddings(5, D)
compare(self_attention(X1 + P), self_attention(X2 + P), "With positional embeddings:")

plt.figure(figsize=(6, 2.5))
plt.imshow(positional_embeddings(50, 16).T, aspect="auto", cmap="RdBu")
plt.xlabel("Position in the sequence")
plt.ylabel("Dimension")
plt.title("Sinusoidal positional embeddings (D = 16)")
plt.colorbar()
plt.show()

**Discussion**

* Why **add** the position vector rather than concatenate it? What are the pros and cons of each option?
* What happens if a test sentence is longer than every sentence seen in training? (Compare sinusoidal and learned embeddings.)
* Does an RNN need positional embeddings? Why or why not?

---
## 3. Multi-head attention

### 3a. Two heads on a tiny example

Each head has its own matrices $W_q^h, W_k^h, W_v^h$, so it can capture a different notion of similarity. The heads' outputs are concatenated and then projected:
$x'_j = [z_j^1 \; z_j^2 \; \cdots \; z_j^H]\, W_0$.

Three words, each a vector of dimension $D = 2$, with two heads. (Convention: row $i$ is the word doing the looking, its **query**; column $j$ is the word being looked at, its **key**.)

In [ ]:
X = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
words = ["the", "cat", "sleeps"]

heads = {
    "head 1": (np.array([[1, 0], [0, 1]], float),    # W_q
               np.array([[1, 0], [1, 1]], float),    # W_k
               np.array([[0, 1], [1, 0]], float)),   # W_v
    "head 2": (np.array([[0, 1], [1, 0]], float),
               np.array([[1, 0], [0, 1]], float),
               np.array([[1, 0], [0, 1]], float)),
}

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
outputs = []
for ax, (name, (Wq_h, Wk_h, Wv_h)) in zip(axes, heads.items()):
    A_h = softmax((X @ Wq_h) @ (X @ Wk_h).T, axis=1)
    outputs.append(A_h @ (X @ Wv_h))
    ax.imshow(A_h, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(words)
    ax.set_yticks(range(3)); ax.set_yticklabels(words)
    ax.set_xlabel("looks at (key)"); ax.set_ylabel("word (query)")
    ax.set_title(name)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{A_h[i, j]:.2f}", ha="center", va="center")
plt.tight_layout()
plt.show()

concat = np.concatenate(outputs, axis=1)               # 3 × (D'·H) = 3 × 4
W0 = np.random.default_rng(1).normal(size=(4, 2))      # (D'·H) × D
print("Concatenated heads:\n", concat)
print("\nMHSA output, x' = [z¹ z²] W0:\n", concat @ W0)

**Discussion:** do the two heads attend to the same words? Why is it useful to have several heads rather than a single, larger one?

### 3b. Exploring the heads of a real model: BERT

BERT (base) is a transformer **encoder**: 12 layers × 12 heads = 144 attention heads.

We look at a "Winograd schema" pair, where a pronoun can only be resolved using the meaning of the sentence:

* "The animal didn't cross the street because **it** was too tired." (it = the animal)
* "The animal didn't cross the street because **it** was too wide." (it = the street)

The two sentences differ by a single word, and the pronoun is the same. Nothing in the grammar tells you what "it" refers to.

**Before exploring:** what kinds of relations do you think different heads might learn? (syntax, the next word, punctuation, coreference…)

The first cell installs BertViz; the next one downloads the model (about 450 MB, a minute or two).

In [ ]:
!pip -q install bertviz

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, attn_implementation="eager")
model.eval()

def attentions(sentence):
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc, output_attentions=True)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    return out.attentions, tokens   # tuple of 12 tensors (1, heads, n, n)

sentence_tired = "The animal didn't cross the street because it was too tired."
sentence_wide = "The animal didn't cross the street because it was too wide."

att_tired, tokens_tired = attentions(sentence_tired)
print("Tokens:", tokens_tired)

Explore freely: pick a layer from the menu, click the colours to isolate heads, and hover over a word to see what it attends to. Look for a head where "it" attends to "animal".

Notice that many heads put most of their attention on the `[CLS]` or `[SEP]` tokens or on punctuation. This is common, and is often interpreted as a way for a head to "do nothing".

In [ ]:
head_view(att_tired, tokens_tired)

### Searching systematically

Rather than scanning 144 heads by eye, we can compute, for each head, how much attention "it" pays to "animal" and to "street" in each sentence.

In [ ]:
def indices(tokens, word):
    return [i for i, t in enumerate(tokens) if t.replace("##", "").lower() == word]

def attention_to(att, tokens, source, target):
    i_src, i_tgt = indices(tokens, source), indices(tokens, target)
    if not i_src or not i_tgt:
        raise ValueError(f"Not found: {source!r} or {target!r} in {tokens}")
    A = torch.stack([a[0] for a in att])      # (layers, heads, n, n)
    return A[:, :, i_src][:, :, :, i_tgt].sum(-1).mean(-1).numpy()   # (layers, heads)

att_wide, tokens_wide = attentions(sentence_wide)
tired_animal = attention_to(att_tired, tokens_tired, "it", "animal")
tired_street = attention_to(att_tired, tokens_tired, "it", "street")
wide_animal  = attention_to(att_wide,  tokens_wide,  "it", "animal")
wide_street  = attention_to(att_wide,  tokens_wide,  "it", "street")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, M, title in [(axes[0], tired_animal, '"it" → "animal" (…too tired)'),
                     (axes[1], wide_street, '"it" → "street" (…too wide)')]:
    im = ax.imshow(M, cmap="viridis")
    ax.set_xlabel("Head"); ax.set_ylabel("Layer"); ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print('The 5 heads where "it" attends most to "animal" in the "tired" sentence:')
for l, h in np.dstack(np.unravel_index(np.argsort(-tired_animal, axis=None)[:5], tired_animal.shape))[0]:
    print(f"  layer {l + 1:>2}, head {h + 1:>2}:  tired: it→animal = {tired_animal[l, h]:.2f}, it→street = {tired_street[l, h]:.2f}"
          f"  |  wide: it→animal = {wide_animal[l, h]:.2f}, it→street = {wide_street[l, h]:.2f}")

**Discussion**

* In the heads that link "it" to "animal" in the "tired" sentence, does the attention shift towards "street" when the last word becomes "wide"? If it doesn't, what might those heads be tracking instead? (The subject of the sentence? The nearest noun?)
* The two sentences are identical up to the last word, and "it" comes *before* that word. Which property of BERT makes it possible for the representation of "it" to depend on "tired" or "wide" at all? Would a left-to-right decoder (like GPT) be able to do the same?
* Does high attention mean the model "uses" that information for its prediction? (Attention shows where information flows, not necessarily what matters for the output.)